# NB5: Kernels GPU personalizados con Numba CUDA

**Computación de Altas Prestaciones para Ciencia de Datos (CAPCD)**

---

## Objetivos de este notebook

1. Entender el modelo de ejecución CUDA: **grids, bloques e hilos**.
2. Escribir *kernels* GPU personalizados con `@cuda.jit`.
3. Configurar correctamente `blocks` y `threads_per_block`.
4. Gestionar memoria GPU explícitamente.
5. Usar *shared memory* para optimizar accesos.
6. Aplicar *grid stride loops* para escalar a datos de cualquier tamaño.
7. Evitar race conditions con operaciones atómicas.
8. Entender el impacto del *memory coalescing* en el rendimiento.

### De CuPy a Numba CUDA
CuPy nos daba operaciones predefinidas (suma, multiplicación, etc.) en GPU. Con `@cuda.jit` podemos programar cualquier lógica que queramos directamente en los miles de núcleos de la GPU.

**Requisito**: GPU activada en Colab.

In [ ]:
!pip install -q numba
!nvidia-smi

---

## 1. El modelo de ejecución CUDA

Cuando lanzamos un *kernel* (función que se ejecuta en la GPU), creamos miles de hilos que ejecutan el mismo código pero sobre datos diferentes.

### Jerarquía

```
Grid (toda la ejecución)
├── Block 0
│   ├── Thread 0
│   ├── Thread 1
│   └── ... (hasta threads_per_block)
├── Block 1
│   ├── Thread 0
│   ├── Thread 1
│   └── ...
└── ... (hasta n_blocks)
```

### Identificación de cada hilo

Cada hilo sabe quién es a través de:
- `cuda.threadIdx.x` — índice del hilo dentro de su bloque (0 a threads_per_block-1)
- `cuda.blockIdx.x` — índice del bloque (0 a n_blocks-1)
- `cuda.blockDim.x` — tamaño del bloque (= threads_per_block)

**Índice global** de un hilo:
```python
i = cuda.threadIdx.x + cuda.blockIdx.x * cuda.blockDim.x
```

### Configuración del lanzamiento

```python
threads_per_block = 256  # Típico: 128, 256, 512
blocks = (N + threads_per_block - 1) // threads_per_block  # Cubrir todos los elementos

mi_kernel[blocks, threads_per_block](args)
```

---

## Demo 1: Tu primer kernel — sumar dos arrays

In [ ]:
from numba import cuda
import numpy as np
import math


@cuda.jit
def suma_arrays_kernel(a, b, resultado):
    """Kernel: cada hilo suma un par de elementos."""
    # Calcular índice global del hilo
    i = cuda.threadIdx.x + cuda.blockIdx.x * cuda.blockDim.x

    # Protección: puede haber más hilos que elementos
    if i < resultado.shape[0]:
        resultado[i] = a[i] + b[i]


# Datos
N = 1_000_000
a_cpu = np.random.rand(N).astype(np.float32)
b_cpu = np.random.rand(N).astype(np.float32)

# Transferir a GPU
a_gpu = cuda.to_device(a_cpu)
b_gpu = cuda.to_device(b_cpu)
resultado_gpu = cuda.device_array(N, dtype=np.float32)  # Array vacío en GPU

# Configurar lanzamiento
threads_per_block = 256
blocks = (N + threads_per_block - 1) // threads_per_block

print(f"Lanzando {blocks} bloques × {threads_per_block} hilos = {blocks * threads_per_block:,} hilos")
print(f"(para {N:,} elementos)\n")

# Lanzar kernel
suma_arrays_kernel[blocks, threads_per_block](a_gpu, b_gpu, resultado_gpu)

# Traer resultado a CPU
resultado_cpu = resultado_gpu.copy_to_host()

# Verificar
esperado = a_cpu + b_cpu
print(f"¿Correcto? {np.allclose(resultado_cpu, esperado)}")
print(f"Primeros 5 resultados: {resultado_cpu[:5]}")
print(f"Esperados:             {esperado[:5]}")

### Anatomía del kernel

```python
@cuda.jit                              # 1. Decorador
def mi_kernel(input, output):          # 2. No devuelve nada (void)
    i = cuda.threadIdx.x + \           # 3. Calcular índice global
        cuda.blockIdx.x * cuda.blockDim.x
    if i < output.shape[0]:            # 4. Protección de límites
        output[i] = input[i] * 2       # 5. Lógica por elemento
```

**Diferencias con funciones normales:**
- No usa `return` — escribe directamente en arrays de salida.
- No tiene bucle `for` — cada hilo procesa un elemento.
- Debe proteger contra accesos fuera de rango.

---

## Demo 2: Kernel práctico, normalización z-score elemento a elemento

In [ ]:
import time


@cuda.jit
def zscore_kernel(data, media, std, resultado):
    """Normalización z-score: resultado[i] = (data[i] - media) / std"""
    i = cuda.threadIdx.x + cuda.blockIdx.x * cuda.blockDim.x
    if i < data.shape[0]:
        resultado[i] = (data[i] - media) / std


N = 20_000_000
data_cpu = np.random.rand(N).astype(np.float32) * 100  # Datos sin normalizar

# Pre-calcular estadísticas en CPU (o CuPy)
media = float(np.mean(data_cpu))
std = float(np.std(data_cpu))

# --- NumPy ---
start = time.perf_counter()
resultado_numpy = (data_cpu - media) / std
t_numpy = time.perf_counter() - start

# --- CUDA Kernel ---
data_gpu = cuda.to_device(data_cpu)
resultado_gpu = cuda.device_array(N, dtype=np.float32)

threads = 256
blocks = (N + threads - 1) // threads

# Warmup
zscore_kernel[blocks, threads](data_gpu, media, std, resultado_gpu)
cuda.synchronize()

start = time.perf_counter()
zscore_kernel[blocks, threads](data_gpu, media, std, resultado_gpu)
cuda.synchronize()
t_cuda = time.perf_counter() - start

# Verificar
resultado_cuda_cpu = resultado_gpu.copy_to_host()
print(f"NumPy:       {t_numpy*1000:.1f} ms")
print(f"CUDA kernel: {t_cuda*1000:.1f} ms")
print(f"Speedup:     {t_numpy/t_cuda:.1f}x")
print(f"¿Correcto?   {np.allclose(resultado_numpy, resultado_cuda_cpu, atol=1e-5)}")

---

## 2. Gestión de memoria GPU

| Función | Acción |
|---|---|
| `cuda.to_device(numpy_array)` | Copia CPU → GPU |
| `cuda.device_array(shape, dtype)` | Crea array vacío en GPU |
| `gpu_array.copy_to_host()` | Copia GPU → CPU |
| `cuda.synchronize()` | Espera a que todos los kernels terminen |

### Patrón típico

```python
# 1. Preparar datos en CPU
datos_cpu = np.random.rand(N)

# 2. Transferir a GPU
datos_gpu = cuda.to_device(datos_cpu)
resultado_gpu = cuda.device_array(N, dtype=np.float32)

# 3. Ejecutar kernel
mi_kernel[blocks, threads](datos_gpu, resultado_gpu)

# 4. Traer resultado
resultado_cpu = resultado_gpu.copy_to_host()
```


---

## Demo 3: Kernel con lógica condicional, filtro de umbral

In [ ]:
@cuda.jit
def filtro_umbral_kernel(data, umbral, resultado):
    """Aplica un filtro: si data[i] > umbral, mantener; si no, poner 0."""
    i = cuda.threadIdx.x + cuda.blockIdx.x * cuda.blockDim.x
    if i < data.shape[0]:
        if data[i] > umbral:
            resultado[i] = data[i]
        else:
            resultado[i] = 0.0


N = 10_000_000
data_cpu = np.random.rand(N).astype(np.float32)
umbral = 0.5

# GPU
data_gpu = cuda.to_device(data_cpu)
resultado_gpu = cuda.device_array(N, dtype=np.float32)

threads = 256
blocks = (N + threads - 1) // threads

filtro_umbral_kernel[blocks, threads](data_gpu, umbral, resultado_gpu)
cuda.synchronize()

resultado_cuda = resultado_gpu.copy_to_host()

# Verificar vs NumPy
resultado_numpy = np.where(data_cpu > umbral, data_cpu, 0.0)
print(f"¿Correcto? {np.allclose(resultado_cuda, resultado_numpy)}")
print(f"Elementos > umbral: {np.count_nonzero(resultado_cuda):,} de {N:,}")

---

## Demo 4: Kernel 2D, multiplicación de matrices

Los kernels CUDA pueden trabajar en 2 dimensiones, útil para matrices e imágenes.


In [ ]:
@cuda.jit
def multiplicar_matrices_kernel(A, B, C):
    """Multiplicación de matrices naive: C = A @ B
    Cada hilo calcula UN elemento de C.
    """
    fila = cuda.threadIdx.x + cuda.blockIdx.x * cuda.blockDim.x
    col = cuda.threadIdx.y + cuda.blockIdx.y * cuda.blockDim.y

    if fila < C.shape[0] and col < C.shape[1]:
        total = 0.0
        for k in range(A.shape[1]):
            total += A[fila, k] * B[k, col]
        C[fila, col] = total


N = 512
A_cpu = np.random.rand(N, N).astype(np.float32)
B_cpu = np.random.rand(N, N).astype(np.float32)

A_gpu = cuda.to_device(A_cpu)
B_gpu = cuda.to_device(B_cpu)
C_gpu = cuda.device_array((N, N), dtype=np.float32)

# Configuración 2D
threads_per_block = (16, 16)  # 16×16 = 256 hilos por bloque
blocks_per_grid = (
    (N + threads_per_block[0] - 1) // threads_per_block[0],
    (N + threads_per_block[1] - 1) // threads_per_block[1],
)

print(f"Grid: {blocks_per_grid} bloques")
print(f"Bloque: {threads_per_block} hilos")
print(f"Total hilos: {blocks_per_grid[0] * blocks_per_grid[1] * 256:,}\n")

# Lanzar
multiplicar_matrices_kernel[blocks_per_grid, threads_per_block](A_gpu, B_gpu, C_gpu)
cuda.synchronize()

C_cuda = C_gpu.copy_to_host()
C_numpy = A_cpu @ B_cpu

print(f"¿Correcto? {np.allclose(C_cuda, C_numpy, atol=1e-3)}")
print(f"Error máximo: {np.max(np.abs(C_cuda - C_numpy)):.6f}")

**Nota:** Esta multiplicación de matrices es *naive* (educativa). CuPy/cuBLAS usan shared memory y tiling para ser mucho más rápidos. Se utiliza como ejemplo de funcionamiento de un kernel 2D.

---

## 3. Shared Memory (concepto avanzado)

Cada bloque tiene acceso a una pequeña zona de memoria ultra-rápida llamada *shared memory* (~48 KB).

```
Memoria global (VRAM) — Grande (~16 GB), lenta (~300 GB/s)
       ↕
Shared memory — Pequeña (~48 KB/bloque), muy rápida (~TB/s)
       ↕
Registros — Mínima, instantánea
```

**Patrón:** Los hilos de un bloque cargan datos de memoria global a shared memory, sincronizan (`cuda.syncthreads()`), y luego trabajan sobre la copia local más rápida.

## Demo 5: Reducción (suma) con shared memory

In [ ]:
@cuda.jit
def suma_reduccion_kernel(data, resultado_parcial):
    """Suma los elementos usando reducción paralela con shared memory."""
    # Shared memory para este bloque
    sdata = cuda.shared.array(256, dtype=np.float32)

    tid = cuda.threadIdx.x
    i = cuda.threadIdx.x + cuda.blockIdx.x * cuda.blockDim.x

    # Cargar a shared memory
    if i < data.shape[0]:
        sdata[tid] = data[i]
    else:
        sdata[tid] = 0.0

    # Barrera: esperar a que todos los hilos del bloque hayan cargado
    cuda.syncthreads()

    # Reducción en árbol
    stride = cuda.blockDim.x // 2
    while stride > 0:
        if tid < stride:
            sdata[tid] += sdata[tid + stride]
        cuda.syncthreads()
        stride //= 2

    # El hilo 0 de cada bloque escribe el resultado parcial
    if tid == 0:
        resultado_parcial[cuda.blockIdx.x] = sdata[0]


N = 1_000_000
data_cpu = np.random.rand(N).astype(np.float32)

data_gpu = cuda.to_device(data_cpu)
threads = 256
blocks = (N + threads - 1) // threads
parciales_gpu = cuda.device_array(blocks, dtype=np.float32)

suma_reduccion_kernel[blocks, threads](data_gpu, parciales_gpu)
cuda.synchronize()

# Sumar los resultados parciales en CPU
parciales_cpu = parciales_gpu.copy_to_host()
suma_cuda = np.sum(parciales_cpu)
suma_numpy = np.sum(data_cpu)

print(f"Suma NumPy:  {suma_numpy:.4f}")
print(f"Suma CUDA:   {suma_cuda:.4f}")
print(f"¿Correcto?   {np.isclose(suma_cuda, suma_numpy, rtol=1e-4)}")

---

## 4. Grid Stride Loops

El *grid stride loop* lanza un grid de tamaño fijo y hace que cada hilo procese múltiples elementos avanzando con un stride igual al tamaño total del grid.

```python
@cuda.jit
def kernel(data, out, N):
    i = cuda.grid(1)            # Índice inicial del hilo
    stride = cuda.gridsize(1)   # Tamaño total del grid
    for idx in range(i, N, stride):  # ← grid stride loop
        out[idx] = data[idx] * 2
```

**Ventajas:**
- Funciona con cualquier tamaño de datos, sin cambiar la configuración de lanzamiento.
- Permite encadenar múltiples pasos de cómputo en **un solo kernel**: los valores intermedios viven en **registros GPU** y nunca se escriben a VRAM entre pasos.
- `cuda.grid(1)` es un atajo de Numba para `threadIdx.x + blockIdx.x * blockDim.x`.

### Por qué los registros importan

```
VRAM (memoria global)  ~1 TB/s    ← acceso en cada lanzamiento separado
Registros GPU          ~100 TB/s  ← acceso dentro del mismo kernel
```

Encadenar N transformaciones en un único kernel stride evita N−1 round-trips a VRAM. Los valores intermedios nunca salen de los registros del hilo.

## Demo 6: Grid stride loop vs kernel simple

In [ ]:
import math

PASSES = 20
_HALF = np.float32(0.5)  # float32 explícito: evita que Numba infiera v como float64

# --- Paso básico: un lanzamiento por paso (enfoque naive) ---
@cuda.jit
def paso_simple(src, dst):
    i = cuda.grid(1)
    if i < dst.shape[0]:
        dst[i] = math.sin(src[i]) * math.cos(src[i]) + _HALF


# --- Grid stride pipeline: PASSES pasos en un único lanzamiento ---
@cuda.jit
def pipeline_stride(data, out, N, passes):
    """Todos los pasos en un solo kernel.
    Los valores intermedios viven en registros GPU float32, no en VRAM.
    """
    i = cuda.grid(1)
    stride = cuda.gridsize(1)
    for idx in range(i, N, stride):
        v = data[idx]                                  # 1 lectura de VRAM (float32)
        for _ in range(passes):
            v = math.sin(v) * math.cos(v) + _HALF     # v permanece float32
        out[idx] = v                                   # 1 escritura a VRAM


N = 20_000_000
data_cpu = np.random.uniform(0.1, 0.9, N).astype(np.float32)
data_gpu = cuda.to_device(data_cpu)

threads = 256
blocks_simple = (N + threads - 1) // threads
num_sm = cuda.get_current_device().MULTIPROCESSOR_COUNT
blocks_stride = num_sm * 8

# --- Naive: PASSES lanzamientos separados ---
buf_a = cuda.to_device(data_cpu.copy())
buf_b = cuda.device_array(N, dtype=np.float32)

paso_simple[blocks_simple, threads](buf_a, buf_b)  # warmup
cuda.synchronize()
buf_a.copy_to_device(data_cpu)  # reset

start = time.perf_counter()
src, dst = buf_a, buf_b
for _ in range(PASSES):
    paso_simple[blocks_simple, threads](src, dst)
    src, dst = dst, src
cuda.synchronize()
t_naive = time.perf_counter() - start
out_naive = src

# --- Stride pipeline: 1 lanzamiento, PASSES pasos en registros ---
out_stride = cuda.device_array(N, dtype=np.float32)
pipeline_stride[blocks_stride, threads](data_gpu, out_stride, N, PASSES)  # warmup
cuda.synchronize()

start = time.perf_counter()
pipeline_stride[blocks_stride, threads](data_gpu, out_stride, N, PASSES)
cuda.synchronize()
t_stride = time.perf_counter() - start

print(f"Naive  ({PASSES} lanzamientos): {t_naive*1000:.2f} ms  ← {PASSES}× leer+escribir VRAM")
print(f"Stride (1 lanzamiento):  {t_stride*1000:.2f} ms  ←  1× leer + {PASSES}× registros + 1× escribir")
print(f"Speedup stride: {t_naive/t_stride:.1f}x")
print(f"¿Mismos resultados? {np.allclose(out_naive.copy_to_host(), out_stride.copy_to_host(), atol=1e-4)}")
print(f"\n★ {PASSES} pasos, pero solo 1 lectura y 1 escritura a VRAM.")
print(f"  Lanzar {PASSES} kernels separados implica {PASSES}× el tráfico de memoria.")

---

## 5. Operaciones atómicas

Cuando múltiples hilos intentan escribir en la misma posición de memoria, hay una *race condition*: el resultado depende del orden de ejecución (impredecible).

```python
# RACE CONDITION: múltiples hilos incrementan contador[0]
@cuda.jit
def contar_mal(data, umbral, contador):
    i = cuda.grid(1)
    if i < data.shape[0] and data[i] > umbral:
        contador[0] += 1  # ← Varios hilos leen, incrementan, escriben al mismo tiempo
```

La solución: operaciones atómicas. Garantizan que la lectura-modificación-escritura sea indivisible.

```python
# CORRECTO: operación atómica
cuda.atomic.add(contador, 0, 1)  # contador[0] += 1, de forma atómica
```

| Operación | Sintaxis |
|---|---|
| Suma atómica | `cuda.atomic.add(array, index, valor)` |
| Máximo atómico | `cuda.atomic.max(array, index, valor)` |
| Mínimo atómico | `cuda.atomic.min(array, index, valor)` |
| Compare-and-swap | `cuda.atomic.compare_and_swap(array, old, new)` |

## Demo 7: Histograma en GPU con operaciones atómicas

In [ ]:
# --- Histograma INCORRECTO (race condition) ---
@cuda.jit
def histograma_mal(data, n_bins, hist):
    i = cuda.grid(1)
    stride = cuda.gridsize(1)
    for idx in range(i, data.shape[0], stride):
        bin_idx = int(data[idx] * n_bins)
        if bin_idx >= n_bins:
            bin_idx = n_bins - 1  # clamp: precisión float32 puede dar exactamente 1.0
        if 0 <= bin_idx < n_bins:
            hist[bin_idx] += 1  # ← RACE CONDITION


# --- Histograma CORRECTO (atómico) ---
@cuda.jit
def histograma_atomico(data, n_bins, hist):
    i = cuda.grid(1)
    stride = cuda.gridsize(1)
    for idx in range(i, data.shape[0], stride):
        bin_idx = int(data[idx] * n_bins)
        if bin_idx >= n_bins:
            bin_idx = n_bins - 1  # clamp: precisión float32 puede dar exactamente 1.0
        if 0 <= bin_idx < n_bins:
            cuda.atomic.add(hist, bin_idx, 1)  # ← ATÓMICO: correcto


# Datos: 10M valores uniformes entre 0 y 1
N = 10_000_000
n_bins = 256
data_cpu = np.random.rand(N).astype(np.float32)
data_gpu = cuda.to_device(data_cpu)

# Referencia NumPy
hist_numpy, _ = np.histogram(data_cpu, bins=n_bins, range=(0, 1))

# Versión con race condition
hist_mal = cuda.device_array(n_bins, dtype=np.int32)
hist_mal.copy_to_device(np.zeros(n_bins, dtype=np.int32))
histograma_mal[128, 256](data_gpu, n_bins, hist_mal)
cuda.synchronize()
hist_mal_cpu = hist_mal.copy_to_host()

# Versión atómica (correcta)
hist_ok = cuda.device_array(n_bins, dtype=np.int32)
hist_ok.copy_to_device(np.zeros(n_bins, dtype=np.int32))
histograma_atomico[128, 256](data_gpu, n_bins, hist_ok)
cuda.synchronize()
hist_ok_cpu = hist_ok.copy_to_host()

print(f"Total NumPy:         {hist_numpy.sum():,}")
print(f"Total race condition: {hist_mal_cpu.sum():,} ← ¡faltan elementos!")
print(f"Total atómico:       {hist_ok_cpu.sum():,} ← correcto")
print(f"\n¿Coincide con NumPy? {np.array_equal(hist_ok_cpu, hist_numpy)}")


---

## 6. Memory coalescing

La GPU lee la memoria global en transacciones de 32, 64 o 128 bytes (segmentos alineados). Los 32 hilos consecutivos de un *warp* acceden a memoria de forma conjunta.

### Acceso coalescido (rápido)
```
Hilos:    [ 0  1  2  3  4  5 ... 31 ]   ← warp
Memoria:  [ 0  1  2  3  4  5 ... 31 ]   ← posiciones contiguas
→ 1 sola transacción de memoria
```

### Acceso strided (lento)
```
Hilos:    [ 0   1   2   3  ...  31 ]   ← warp
Memoria:  [ 0  16  32  48  ... 496 ]   ← posiciones separadas
→ Múltiples transacciones de memoria (hasta 32×)
```

Como regla general, hilos consecutivos deben acceder a posiciones de memoria consecutivas.

### Implicación para matrices 2D
Las matrices en C/NumPy se almacenan por filas (*row-major*). Si los hilos recorren las columnas de una fila (eje 1), el acceso es coalescido. Si recorren las filas de una columna (eje 0), es strided.


## Demo 8: Impacto del memory coalescing

In [ ]:
@cuda.jit
def suma_filas_kernel(matrix, sums, n_cols):
    """Cada hilo suma una fila completa.
    Hilos consecutivos acceden a FILAS distintas → acceso a columna 0 NO es contiguo.
    """
    i = cuda.grid(1)
    if i < sums.shape[0]:
        total = 0.0
        for j in range(n_cols):
            total += matrix[i, j]  # hilos 0..31 acceden a matrix[0..31, j] (strided)
        sums[i] = total


@cuda.jit
def suma_columnas_kernel(matrix, sums, n_rows):
    """Cada hilo suma una columna completa.
    Hilos consecutivos acceden a COLUMNAS contiguas → ¡acceso coalescido!
    """
    j = cuda.grid(1)
    if j < sums.shape[0]:
        total = 0.0
        for i in range(n_rows):
            total += matrix[i, j]  # hilos 0..31 acceden a matrix[i, 0..31] (contiguo)
        sums[j] = total


# Matriz cuadrada grande
N = 8192
matrix_cpu = np.ones((N, N), dtype=np.float32)
matrix_gpu = cuda.to_device(matrix_cpu)

threads = 256
blocks = (N + threads - 1) // threads

# Suma por filas (acceso NO coalescido)
sums_filas = cuda.device_array(N, dtype=np.float32)
suma_filas_kernel[blocks, threads](matrix_gpu, sums_filas, N)  # warmup
cuda.synchronize()

start = time.perf_counter()
for _ in range(10):
    suma_filas_kernel[blocks, threads](matrix_gpu, sums_filas, N)
cuda.synchronize()
t_filas = (time.perf_counter() - start) / 10

# Suma por columnas (acceso coalescido)
sums_cols = cuda.device_array(N, dtype=np.float32)
suma_columnas_kernel[blocks, threads](matrix_gpu, sums_cols, N)  # warmup
cuda.synchronize()

start = time.perf_counter()
for _ in range(10):
    suma_columnas_kernel[blocks, threads](matrix_gpu, sums_cols, N)
cuda.synchronize()
t_cols = (time.perf_counter() - start) / 10

print(f"Suma filas (NO coalescido):  {t_filas*1000:.2f} ms")
print(f"Suma columnas (coalescido):  {t_cols*1000:.2f} ms")
print(f"Speedup coalescido:          {t_filas/t_cols:.1f}x")
print(f"\n¿Ambos correctos? filas={np.allclose(sums_filas.copy_to_host(), N)}, "
      f"cols={np.allclose(sums_cols.copy_to_host(), N)}")
print("\n★ Mismo trabajo, mismos datos, diferente patrón de acceso → gran diferencia.")

---

## Ejercicio 1: Kernel de operación elemento a elemento

Escribe un kernel que calcule $\text{resultado}[i] = \sin(a[i])^2 + \cos(a[i])^2$ para verificar la identidad trigonométrica (debería dar 1.0 para todos los elementos).


In [ ]:
from numba import cuda
import numpy as np
import math


@cuda.jit
def identidad_trig_kernel(a, resultado):
    """Calcula sin(a[i])^2 + cos(a[i])^2 para cada elemento.
    Debería dar 1.0 siempre.
    """
    i = cuda.threadIdx.x + cuda.blockIdx.x * cuda.blockDim.x
    # TODO: Implementa
    # Pista: usa math.sin() y math.cos() (no np.sin)
    # No olvides la protección de límites: if i < ...


N = 5_000_000
a_cpu = np.random.rand(N).astype(np.float64) * 2 * np.pi

# TODO:
# 1. Transfiere a_cpu a GPU
# 2. Crea resultado_gpu con cuda.device_array()
# 3. Configura threads y blocks
# 4. Lanza el kernel
# 5. Trae resultado a CPU

resultado_cpu = None  # TODO: reemplaza

In [ ]:
# Autoevaluación
assert resultado_cpu is not None, "Debes calcular el resultado"
assert np.allclose(resultado_cpu, 1.0, atol=1e-10), "sin²(x) + cos²(x) debe ser 1.0"
print("✅ ¡Correcto! La identidad trigonométrica se cumple en GPU.")

---

## Ejercicio 2: Kernel para distancia euclídea por filas

Dada una matriz de puntos y un punto de referencia, escribe un kernel que calcule la **distancia euclídea** de cada fila al punto de referencia.

$$d_i = \sqrt{\sum_{j=0}^{D-1} (\text{puntos}[i,j] - \text{ref}[j])^2}$$

Cada **hilo** procesa una **fila completa**.

In [ ]:
@cuda.jit
def distancias_kernel(puntos, referencia, distancias):
    """Cada hilo calcula la distancia de una fila al punto de referencia."""
    i = cuda.threadIdx.x + cuda.blockIdx.x * cuda.blockDim.x

    if i < puntos.shape[0]:
        # TODO:
        # 1. Acumula sum((puntos[i,j] - referencia[j])^2) para j en range(n_dims)
        # 2. distancias[i] = sqrt del acumulado
        # Pista: usa math.sqrt()
        pass


N_PUNTOS = 1_000_000
N_DIMS = 64

puntos_cpu = np.random.rand(N_PUNTOS, N_DIMS).astype(np.float32)
ref_cpu = np.random.rand(N_DIMS).astype(np.float32)

# TODO:
# 1. Transfiere puntos y ref a GPU
# 2. Crea array de distancias en GPU
# 3. Configura y lanza kernel
# 4. Trae resultado a CPU

distancias_cuda = None  # TODO: reemplaza

# Verificar con NumPy
distancias_numpy = np.sqrt(np.sum((puntos_cpu - ref_cpu) ** 2, axis=1))

In [ ]:
# Autoevaluación
assert distancias_cuda is not None, "Debes calcular las distancias"
assert distancias_cuda.shape == (N_PUNTOS,), f"Shape incorrecto: {distancias_cuda.shape}"
assert np.allclose(distancias_cuda, distancias_numpy, rtol=1e-4), "Las distancias no coinciden"
print("✅ ¡Correcto! Distancias calculadas en GPU correctamente.")

---

## Ejercicio 3: Kernel de mapa de calor (2D)

Escribe un kernel 2D que genere un **mapa de calor** basado en la distancia de cada píxel al centro.

$$\text{heatmap}[y, x] = e^{-\frac{(x - cx)^2 + (y - cy)^2}{2\sigma^2}}$$

donde $(cx, cy)$ es el centro y $\sigma$ controla la dispersión.

In [ ]:
@cuda.jit
def heatmap_kernel(output, cx, cy, sigma):
    """Genera un mapa de calor gaussiano.
    Cada hilo calcula un píxel.
    """
    # TODO: Calcula coordenadas 2D del hilo
    # x = cuda.threadIdx.x + cuda.blockIdx.x * cuda.blockDim.x
    # y = cuda.threadIdx.y + cuda.blockIdx.y * cuda.blockDim.y
    # Protección de límites: if x < output.shape[1] and y < output.shape[0]
    # Fórmula: output[y, x] = math.exp(-(dx*dx + dy*dy) / (2 * sigma * sigma))
    pass


H, W = 1024, 1024
output_gpu = cuda.device_array((H, W), dtype=np.float32)

threads = (16, 16)
blocks = (
    (W + threads[0] - 1) // threads[0],
    (H + threads[1] - 1) // threads[1],
)

# Centro en el medio, sigma = 200
heatmap_kernel[blocks, threads](output_gpu, W / 2, H / 2, 200.0)
cuda.synchronize()

heatmap_cpu = output_gpu.copy_to_host()

In [ ]:
# Visualizar
import matplotlib.pyplot as plt

plt.figure(figsize=(8, 8))
plt.imshow(heatmap_cpu, cmap='hot', origin='lower')
plt.colorbar(label='Intensidad')
plt.title('Mapa de calor gaussiano (generado en GPU)')
plt.show()

In [ ]:
# Autoevaluación
assert heatmap_cpu.shape == (1024, 1024), "Shape incorrecto"
assert heatmap_cpu[512, 512] > 0.99, "El centro debe tener valor ~1.0"
assert heatmap_cpu[0, 0] < 0.01, "Las esquinas deben tener valor ~0.0"
print("✅ ¡Correcto! Mapa de calor gaussiano generado en GPU.")

---

## Resumen

| Concepto | Detalle |
|---|---|
| **Kernel** | Función que ejecutan miles de hilos en paralelo |
| **Grid/Block/Thread** | Jerarquía de organización de hilos |
| **`@cuda.jit`** | Decorador para crear kernels |
| **`cuda.threadIdx.x`** | Índice local del hilo |
| **`cuda.blockIdx.x`** | Índice del bloque |
| **`cuda.to_device()`** | CPU → GPU |
| **`cuda.device_array()`** | Crear array en GPU |
| **`cuda.syncthreads()`** | Barrera dentro de un bloque |
| **Shared memory** | Memoria ultra-rápida compartida por bloque |
| **Grid stride loop** | Cada hilo procesa múltiples elementos con `range(i, N, stride)` |
| **`cuda.atomic.add()`** | Escritura atómica para evitar race conditions |
| **Memory coalescing** | Hilos consecutivos → memoria contigua = rápido |

### Cuándo usar Numba CUDA vs CuPy

| Situación | Herramienta |
|---|---|
| Operación estándar (suma, multiplicación, FFT...) | **CuPy** (más simple, más optimizado) |
| Lógica personalizada por elemento | **Numba CUDA** |
| Algoritmo con datos compartidos entre hilos | **Numba CUDA** con shared memory |
| Histogramas, contadores GPU | **Numba CUDA** con operaciones atómicas |